In [ ]:
#Notebook description

# this notebook is intended to track and price commodities based on the demand and supply data

In [ ]:
# Databento connection setup
import os
import sys
import subprocess
import getpass

try:
    import databento as db
except ImportError:
    # Install into the active notebook kernel environment (not terminal env).
    subprocess.check_call([sys.executable, "-m", "pip", "install", "databento", "--quiet"])
    import databento as db

API_ENV_VAR = "DATABENTO_API_KEY"
ALLOW_INTERACTIVE_PROMPT = False  # Set True only if you want a manual prompt fallback.

DATABENTO_API_KEY = os.getenv(API_ENV_VAR, "").strip()

# Try OS keyring next so you don't have to retype keys in notebooks.
if not DATABENTO_API_KEY:
    try:
        import keyring
        DATABENTO_API_KEY = keyring.get_password("databento", "default") or ""
    except Exception:
        DATABENTO_API_KEY = ""

if not DATABENTO_API_KEY and ALLOW_INTERACTIVE_PROMPT:
    DATABENTO_API_KEY = getpass.getpass("Enter DATABENTO_API_KEY: ").strip()

if not DATABENTO_API_KEY:
    raise ValueError(
        "DATABENTO_API_KEY not found. Set env var DATABENTO_API_KEY, or save to keyring under service='databento', username='default'."
    )

# Keep it available in this kernel session.
os.environ[API_ENV_VAR] = DATABENTO_API_KEY

db_client = db.Historical(DATABENTO_API_KEY)
print(f"Databento historical client initialized with kernel: {sys.executable}")

In [ ]:
# Databento one-query CL curve loader (daily bars, cost-aware)
from datetime import date, timedelta

TEST_DATASET = "GLBX.MDP3"
SCHEMA = "ohlcv-1d"
ROOT = "CL"

# Curve depth and window controls (single query only).
N_CURVE_CONTRACTS = 24   # target 24 monthly CL contracts
DAYS_BACK = 60           # longer lookback helps include thinly traded deferred months

if N_CURVE_CONTRACTS > 24:
    raise ValueError("Cost guard: keep N_CURVE_CONTRACTS <= 24.")
if DAYS_BACK > 93:
    raise ValueError("Cost guard: keep DAYS_BACK <= 93.")

# CL lists monthly contracts (all month codes).
month_cycle = [(1, "F"), (2, "G"), (3, "H"), (4, "J"), (5, "K"), (6, "M"), (7, "N"), (8, "Q"), (9, "U"), (10, "V"), (11, "X"), (12, "Z")]
today = date.today()

# Find the first contract month at or after current month.
start_idx = 0
for i, (m, _) in enumerate(month_cycle):
    if today.month <= m:
        start_idx = i
        break
else:
    start_idx = 0

TEST_SYMBOLS = []
year = today.year
idx = start_idx
for _ in range(N_CURVE_CONTRACTS):
    month_num, month_code = month_cycle[idx]
    if idx == 0 and month_num < today.month:
        year += 1
    yy = year % 10  # Databento raw symbol style, e.g., CLM6
    TEST_SYMBOLS.append(f"{ROOT}{month_code}{yy}")
    idx += 1
    if idx >= len(month_cycle):
        idx = 0
        year += 1

end_date = today
start_date = end_date - timedelta(days=DAYS_BACK)

print(
    f"Single Databento query: {len(TEST_SYMBOLS)} symbols, schema={SCHEMA}, "
    f"window={start_date} to {end_date}"
)
print(f"Symbols requested: {TEST_SYMBOLS}")

# One query for the entire requested curve.
bars = db_client.timeseries.get_range(
    dataset=TEST_DATASET,
    schema=SCHEMA,
    symbols=TEST_SYMBOLS,
    stype_in="raw_symbol",
    start=start_date.isoformat(),
    end=end_date.isoformat(),
)

daily_df = bars.to_df()

if daily_df.empty:
    print("No rows returned. Try confirming dataset access.")
else:
    tmp = daily_df.copy().reset_index()
    sym_col = "symbol" if "symbol" in tmp.columns else "raw_symbol"
    returned_symbols = sorted(set(tmp[sym_col].astype(str))) if sym_col in tmp.columns else []
    missing = sorted(set(TEST_SYMBOLS) - set(returned_symbols))
    print(f"Rows returned: {len(daily_df):,}")
    print(f"Contracts requested: {len(TEST_SYMBOLS)} | with data: {len(returned_symbols)}")
    if missing:
        print(f"No rows for: {missing}")
    display(daily_df.tail(10))

In [ ]:
# Plot all retrieved CL data from daily_df (no Databento re-query)
import pandas as pd
import plotly.graph_objects as go

if "daily_df" not in globals() or daily_df.empty:
    raise ValueError("daily_df is missing or empty. Run the Databento pull cell first.")

plot_df = daily_df.copy().reset_index()

# Handle common Databento DataFrame shapes.
symbol_col = "symbol" if "symbol" in plot_df.columns else "raw_symbol"
if symbol_col not in plot_df.columns:
    raise ValueError("Could not find a symbol column in daily_df.")
if "close" not in plot_df.columns:
    raise ValueError("daily_df does not contain a 'close' column.")

if "ts_event" in plot_df.columns:
    plot_df["asof_ts"] = pd.to_datetime(plot_df["ts_event"], utc=True)
elif "date" in plot_df.columns:
    plot_df["asof_ts"] = pd.to_datetime(plot_df["date"], utc=True)
else:
    raise ValueError("Could not find a date field (expected 'ts_event' or 'date').")

plot_df = plot_df.sort_values([symbol_col, "asof_ts"]).reset_index(drop=True)

fig = go.Figure()
for sym, grp in plot_df.groupby(symbol_col):
    fig.add_trace(
        go.Scatter(
            x=grp["asof_ts"],
            y=grp["close"],
            mode="lines+markers",
            name=str(sym),
            hovertemplate="Symbol: " + str(sym) + "<br>Date: %{x|%Y-%m-%d}<br>Close: %{y:.2f}<extra></extra>",
        )
    )

fig.update_layout(
    title="CL Retrieved Daily Data (all rows from current pull)",
    xaxis_title="Date",
    yaxis_title="Daily Close",
    template="plotly_white",
    height=560,
)

fig.show()

print(f"Total rows plotted: {len(plot_df):,}")
print(f"Unique contracts plotted: {plot_df[symbol_col].nunique()}")
display(plot_df[[symbol_col, "asof_ts", "close"]].reset_index(drop=True))

In [ ]:
# Plot CL futures curve from existing daily_df (no Databento re-query)
import re
import pandas as pd
import plotly.graph_objects as go

if "daily_df" not in globals() or daily_df.empty:
    raise ValueError("daily_df is missing or empty. Run the Databento pull cell first.")

# If True, require exact today's date rows only (may produce fewer points).
STRICT_TODAY = False

curve_df = daily_df.copy().reset_index()
symbol_col = "symbol" if "symbol" in curve_df.columns else "raw_symbol"
if symbol_col not in curve_df.columns:
    raise ValueError("Could not find a symbol column in daily_df.")
if "close" not in curve_df.columns:
    raise ValueError("daily_df does not contain a 'close' column.")

if "ts_event" in curve_df.columns:
    curve_df["asof_ts"] = pd.to_datetime(curve_df["ts_event"], utc=True)
elif "date" in curve_df.columns:
    curve_df["asof_ts"] = pd.to_datetime(curve_df["date"], utc=True)
else:
    raise ValueError("Could not find a date field (expected 'ts_event' or 'date').")

today_utc = pd.Timestamp.utcnow().normalize()

if STRICT_TODAY:
    curve_rows = curve_df[curve_df["asof_ts"].dt.normalize() == today_utc].copy()
    asof_label = f"today (UTC): {today_utc.date()}"
else:
    # Default: one latest available point per contract, so you get the fullest curve from one query.
    curve_rows = (
        curve_df.sort_values([symbol_col, "asof_ts"])
        .groupby(symbol_col, as_index=False)
        .tail(1)
        .copy()
    )
    asof_label = "latest available per contract in current pull"

if curve_rows.empty:
    raise ValueError("No contract rows available to plot the curve.")

# Parse CL month/year for proper curve ordering.
month_order = {"F":1, "G":2, "H":3, "J":4, "K":5, "M":6, "N":7, "Q":8, "U":9, "V":10, "X":11, "Z":12}
pat = re.compile(r"([FGHJKMNQUVXZ])(\d{1,2})$")

def sort_key_for_contract(sym: str):
    m = pat.search(str(sym))
    if not m:
        return None
    mcode = m.group(1)
    yy = int(m.group(2))
    year = 2000 + yy if yy < 80 else 1900 + yy
    month = month_order[mcode]
    return year * 100 + month

curve_rows["sort_key"] = curve_rows[symbol_col].map(sort_key_for_contract)
curve_rows = curve_rows.dropna(subset=["sort_key"]).sort_values("sort_key").reset_index(drop=True)

if curve_rows.empty:
    raise ValueError("No parsable contract symbols available to plot the curve.")

# Staleness helps identify deferred contracts that have older last trade dates.
curve_rows["age_days"] = (today_utc - curve_rows["asof_ts"].dt.normalize()).dt.days

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=curve_rows[symbol_col],
        y=curve_rows["close"],
        mode="lines+markers",
        name="CL curve",
        customdata=curve_rows[["asof_ts", "age_days"]],
        hovertemplate="Contract: %{x}<br>Close: %{y:.2f}<br>Last Date: %{customdata[0]|%Y-%m-%d}<br>Age: %{customdata[1]}d<extra></extra>",
    )
)

fig.update_layout(
    title=f"CL Futures Curve ({asof_label})",
    xaxis_title="Contract Month",
    yaxis_title="Daily Close",
    template="plotly_white",
    height=520,
)

fig.show()
print(f"Contracts on curve: {len(curve_rows)}")
display(curve_rows[[symbol_col, "asof_ts", "age_days", "close"]])

In [ ]:
#Load libraries 
import logging
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False
import sys
sys.path.append(r"e:\Coding Projects\Investment Analysis")

# Load libraries
from Quantapp.visualization import Plotter
from Quantapp.analytics import TimeSeriesAnalytics as Rolling
from Quantapp.data import MacroDataClient

import numpy as np
import json
import os
import pandas as pd
import yfinance as yf
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.graph_objects as go
import plotly.graph_objects as go
import pandas as pd
from plotly.subplots import make_subplots

#shut down warnings
import warnings
warnings.filterwarnings("ignore")


qc = Rolling()
qp = Plotter()
qe = MacroDataClient()

In [ ]:
#parameters
period = "20y"

In [ ]:
#Load parameters
market_benchmark = yf.Ticker("SPY").history(period=period)
commodities_benchmark_broad = yf.Ticker("DBC").history(period=period)
commodities_benchmark_energy = yf.Ticker("DBE").history(period=period)
commodities_benchmark_industrial_metals = yf.Ticker("DBB").history(period=period)
commodities_benchmark_precious_metals = yf.Ticker("DBP").history(period=period)
commodities_benchmark_rare_earth_metals = yf.Ticker("EVMT").history(period=period)
commodities_benchmark_agriculture = yf.Ticker("DBA").history(period=period)

commodities_dashboard = {
    "Agriculture": {
        "Grains": {
            "Wheat": "ZW=F",
            "Corn": "ZC=F",
            "Soybeans": "ZS=F",
            "Soybean Oil": "ZL=F",
            "Soybean Meal": "ZM=F",
            "Oats": "ZO=F",
            "Rough Rice": "ZR=F"
        },
        "Softs": {
            "Coffee": "KC=F",
            "Cocoa": "CC=F",
            "Sugar": "SB=F",
            "Cotton": "CT=F",
            "Orange Juice": "OJ=F"
        },
        "Livestock": {
            "Live Cattle": "LE=F",
            "Feeder Cattle": "GF=F",
            "Lean Hogs": "HE=F"
        }
    },
    "Energy": {
        "Crude Oil": "CL=F",
        "Natural Gas": "NG=F",
        "Heating Oil": "HO=F",
        "Gasoline": "RB=F"
    },
    "Industrial / Construction Materials": {
        "Copper": "HG=F",
        "Aluminum": "ALI=F",   # optional ETF for tracking if needed
        "Zinc": "ZN=F",
        "Lumber": "LBR=F"       # housing/construction cycle commodity
    },
    "Tech / Battery / Rare Earth Metals": {
        "Lithium ETF": "LIT",
        "Rare Earth ETF": "REMX"
    },
    "Precious Metals": {
        "Gold": "GC=F",
        "Silver": "SI=F",
        "Platinum": "PL=F",
        "Palladium": "PA=F"
    }
}


# Extract dictionaries from commodities_dashboard
commodities_dict_energy = commodities_dashboard["Energy"]
commodities_dict_industrial_metals = commodities_dashboard["Industrial / Construction Materials"]
commodities_rare_earth_metals = commodities_dashboard["Tech / Battery / Rare Earth Metals"]
commodities_dict_precious_metals = commodities_dashboard["Precious Metals"]
commodities_dict_agriculture_grains = commodities_dashboard["Agriculture"]["Grains"]
commodities_livestock = commodities_dashboard["Agriculture"]["Livestock"]
commodities_dict_agriculture_softs = commodities_dashboard["Agriculture"]["Softs"]

#turn commodities_dict_energy into a dataframe with closing prices
commodities_df_energy = pd.DataFrame()

for key, value in commodities_dict_energy.items():
    commodities_df_energy[key] = yf.Ticker(value).history(period=period)["Close"]
    
#turn commodities_dict_industrial_metals into a dataframe with closing prices
commodities_df_metals = pd.DataFrame()

for key, value in commodities_dict_industrial_metals.items():
    commodities_df_metals[key] = yf.Ticker(value).history(period=period)["Close"]
    
for key, value in commodities_rare_earth_metals.items():
    commodities_df_metals[key] = yf.Ticker(value).history(period=period)["Close"]


#turn commodities_dict_precious_metals into a dataframe with closing prices
commodities_df_precious_metals = pd.DataFrame()

for key, value in commodities_dict_precious_metals.items():
    commodities_df_precious_metals[key] = yf.Ticker(value).history(period=period)["Close"]
    
#turn commodities_dict_agriculture_grains into a dataframe with closing prices
commodities_df_grains = pd.DataFrame()

for key, value in commodities_dict_agriculture_grains.items():
    commodities_df_grains[key] = yf.Ticker(value).history(period=period)["Close"]

#turn commodities_livestock into a dataframe with closing prices
commodities_df_livestock = pd.DataFrame()

for key, value in commodities_livestock.items():
    commodities_df_livestock[key] = yf.Ticker(value).history(period=period)["Close"]

#turn commodities_dict_agriculture_softs into a dataframe with closing prices
commodities_df_softs = pd.DataFrame()

for key, value in commodities_dict_agriculture_softs.items():
    commodities_df_softs[key] = yf.Ticker(value).history(period=period)["Close"]

time_frame_week = 7
time_frame_short = 21
time_frame_mid   = 50
time_frame_long = 200

#plot all commodities benchmarks



In [ ]:
#plot broad commodities benchmark
fig = make_subplots(rows=1, cols=1, subplot_titles=["Commodities Benchmarks"])
fig.add_trace(go.Scatter(x=commodities_benchmark_broad.index, y=commodities_benchmark_broad['Close'], mode='lines', name='Broad Commodities (DBC)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_benchmark_energy.index, y=commodities_benchmark_energy['Close'], mode='lines', name='Energy Commodities (DBE)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_benchmark_industrial_metals.index, y=commodities_benchmark_industrial_metals['Close'], mode='lines', name='Industrial Metals (DBB)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_benchmark_precious_metals.index, y=commodities_benchmark_precious_metals['Close'], mode='lines', name='Precious Metals (DBP)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_benchmark_rare_earth_metals.index, y=commodities_benchmark_rare_earth_metals['Close'], mode='lines', name='Rare Earth Metals (EVMT)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_benchmark_agriculture.index, y=commodities_benchmark_agriculture['Close'], mode='lines', name='Agriculture Commodities (DBA)'), row=1, col=1)
fig.update_layout(height=1200, title_text="Commodities Benchmarks Over Time")
fig.show()

In [ ]:
#plot energy commodities
fig = make_subplots(rows=1, cols=1, subplot_titles=["Energy Commodities"])
fig.add_trace(go.Scatter(x=commodities_df_energy.index, y=commodities_df_energy['Crude Oil'], mode='lines', name='Crude Oil (CL=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_energy.index, y=commodities_df_energy['Natural Gas'], mode='lines', name='Natural Gas (NG=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_energy.index, y=commodities_df_energy['Heating Oil'], mode='lines', name='Heating Oil (HO=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_energy.index, y=commodities_df_energy['Gasoline'], mode='lines', name='Gasoline (RB=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Energy Commodities Over Time")
fig.show()

In [ ]:
#plot industrial metals commodities
fig = make_subplots(rows=1, cols=1, subplot_titles=["Industrial / Construction Materials Commodities"])
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Copper'], mode='lines', name='Copper (HG=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Aluminum'], mode='lines', name='Aluminum (ALI=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Zinc'], mode='lines', name='Zinc (ZN=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Lumber'], mode='lines', name='Lumber (LBR=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Industrial / Construction Materials Commodities Over Time")
fig.show()


In [ ]:
#plot rare earth metals commodities
fig = make_subplots(rows=1, cols=1, subplot_titles=["Tech / Battery / Rare Earth Metals Commodities"])
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Lithium ETF'], mode='lines', name='Lithium ETF (LIT)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_metals.index, y=commodities_df_metals['Rare Earth ETF'], mode='lines', name='Rare Earth ETF (REMX)'), row=1, col=1)
fig.update_layout(height=600, title_text="Tech / Battery / Rare Earth Metals Commodities Over Time")
fig.show()


In [ ]:
#plot precious metals commodities
fig = make_subplots(rows=1, cols=1, subplot_titles=["Precious Metals Commodities"])
fig.add_trace(go.Scatter(x=commodities_df_precious_metals.index, y=commodities_df_precious_metals['Gold'], mode='lines', name='Gold (GC=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_precious_metals.index, y=commodities_df_precious_metals['Silver'], mode='lines', name='Silver (SI=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_precious_metals.index, y=commodities_df_precious_metals['Platinum'], mode='lines', name='Platinum (PL=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_precious_metals.index, y=commodities_df_precious_metals['Palladium'], mode='lines', name='Palladium (PA=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Precious Metals Commodities Over Time")
fig.show()


In [ ]:
#plot agriculture commodities - grains    
fig = make_subplots(rows=1, cols=1, subplot_titles=["Agriculture Commodities - Grains"])
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Wheat'], mode='lines', name='Wheat (ZW=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Corn'], mode='lines', name='Corn (ZC=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Soybeans'], mode='lines', name='Soybeans (ZS=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Soybean Oil'], mode='lines', name='Soybean Oil (ZL=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Soybean Meal'], mode='lines', name='Soybean Meal (ZM=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Oats'], mode='lines', name='Oats (ZO=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_grains.index, y=commodities_df_grains['Rough Rice'], mode='lines', name='Rough Rice (ZR=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Agriculture Commodities - Grains Over Time")

fig.show()

In [ ]:
#plot agriculture commodities - softs
fig = make_subplots(rows=1, cols=1, subplot_titles=["Agriculture Commodities - Softs"])
fig.add_trace(go.Scatter(x=commodities_df_softs.index, y=commodities_df_softs['Coffee'], mode='lines', name='Coffee (KC=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_softs.index, y=commodities_df_softs['Cocoa'], mode='lines', name='Cocoa (CC=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_softs.index, y=commodities_df_softs['Sugar'], mode='lines', name='Sugar (SB=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_softs.index, y=commodities_df_softs['Cotton'], mode='lines', name='Cotton (CT=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_softs.index, y=commodities_df_softs['Orange Juice'], mode='lines', name='Orange Juice (OJ=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Agriculture Commodities - Softs Over Time")
fig.show()


In [ ]:
#plot agriculture commodities - livestock
fig = make_subplots(rows=1, cols=1, subplot_titles=["Agriculture Commodities - Livestock"])
fig.add_trace(go.Scatter(x=commodities_df_livestock.index, y=commodities_df_livestock['Live Cattle'], mode='lines', name='Live Cattle (LE=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_livestock.index, y=commodities_df_livestock['Feeder Cattle'], mode='lines', name='Feeder Cattle (GF=F)'), row=1, col=1)
fig.add_trace(go.Scatter(x=commodities_df_livestock.index, y=commodities_df_livestock['Lean Hogs'], mode='lines', name='Lean Hogs (HE=F)'), row=1, col=1)
fig.update_layout(height=600, title_text="Agriculture Commodities - Livestock Over Time")
fig.show()
